In [ ]:
import pyguidos
from pyguidos import data

import rasterio
import numpy as np
import os
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.colors import ListedColormap, BoundaryNorm

In [ ]:
data_dir = data.test_data_dir()

in_dir = data_dir / 'binarymap'
out_dir = Path('/home/user/tmp/output/') # <<< REPLACE with your actual desired output path
                                         #     The folder must be empty

In [ ]:
bin_map = input_dir / 'binarymap.tif'

with rasterio.open(bin_map) as src:
    array = src.read(1)

fig, ax = plt.subplots(figsize=(10, 8))

colors = ['white', 'lightgrey', 'darkgreen'] 
cmap = ListedColormap(colors)
im = ax.imshow(array, cmap=cmap, vmin=-0.5, vmax=2.5)
legend_labels = {
    0: 'Missing/NoData', 
    1: 'Background',     
    2: 'Foreground'      
}

patches = []
for val in sorted(legend_labels.keys()): 
    label = legend_labels[val]
    color = colors[val] 
    patch = mpatches.Patch(color=color, label=label)
    patches.append(patch)
legend = ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.axis('off')
plt.show()

In [ ]:

pyguidos.gwb_frag(in_dir, out_dir, conn_8=True, pix_res=100, window_size=27, method='FAD_6', stats=True)

In [ ]:
map_name = str(bin_map).split('/')[-1][:-4]
res_dir = os.path.join(out_dir, map_name + '_frag')

frag_map = os.path.join(res_dir, map_name + '_fos-fad_6class_27.tif')
frag_map_png = os.path.join(res_dir, map_name + '_fos-fad_6class_27.png')
frag_map_csv = os.path.join(res_dir, map_name + '_fos-fad_6class.csv')
frag_map_txt = os.path.join(res_dir, map_name + '_fos-fad_6class.txt')

In [ ]:
with rasterio.open(frag_map) as src:
    array = src.read(1)

class_scheme_6class = [
    {'name': 'Rare', 'min_val': 0, 'max_val': 9, 'color': '#d63127', 'label': 'Rare [0-9]'},         
    {'name': 'Patchy', 'min_val': 10, 'max_val': 39, 'color': '#f98b59', 'label': 'Patchy [10-39]'}, 
    {'name': 'Transitional', 'min_val': 40, 'max_val': 59, 'color': '#fec700', 'label': 'Transitional [40-59]'}, 
    {'name': 'Dominant', 'min_val': 60, 'max_val': 89, 'color': '#8bc763', 'label': 'Dominant [60-89]'}, 
    {'name': 'Interior', 'min_val': 90, 'max_val': 99, 'color': '#00ae00', 'label': 'Interior [90-99]'}, 
    {'name': 'Intact', 'min_val': 100, 'max_val': 100, 'color': '#007700', 'label': 'Intact (100)'}, 
    {'name': 'Background', 'min_val': 101, 'max_val': 101, 'color': '#aeaeae', 'label': 'Background (101)'}, 
    {'name': 'NoData', 'min_val': 102, 'max_val': 102, 'color': 'white', 'label': 'No Data (102)'}            
]

norm_boundaries = []
cmap_colors = []
value_to_category_map = {}
class_scheme_6class.sort(key=lambda x: x['min_val'])
for i, category in enumerate(class_scheme_6class):
    cmap_colors.append(category['color'])
    if i == 0:
        norm_boundaries.append(category['min_val'] - 0.5)
    norm_boundaries.append(category['max_val'] + 0.5)
    for val in range(category['min_val'], category['max_val'] + 1):
        value_to_category_map[val] = category['name']
cmap = ListedColormap(cmap_colors)
norm = BoundaryNorm(norm_boundaries, cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(array, cmap=cmap, norm=norm, interpolation='nearest')

unique_values = np.unique(array).tolist()
categories = set()
for val in unique_values:
    if val in value_to_category_map:
        categories.add(value_to_category_map[val])

patches = []
for category in class_scheme_6class:
    if category['name'] in categories:
        patch = mpatches.Patch(color=category['color'], label=category['label'])
        patches.append(patch)
legend = ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

plt.axis('off')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
img = mpimg.imread(frag_map_png)
ax.imshow(img)
ax.axis('off')
plt.show()

In [ ]:
with open(frag_map_txt, 'r', encoding='utf-8') as f:
    txt = f.read()
    print(txt)

In [ ]:
with open(frag_map_csv, 'r', encoding='utf-8') as f:
    txt = f.read()
    print(txt)